# UCI Observation-Group Signal Validation

Validates that predictive information is highly concentrated in the 10-variable Observation group, 
including Observation-only, non-Observation-only, top-2, remaining-8, and leave-one-variable-out analyses.
This notebook expects outputs produced by `01_uci_main_experiments.ipynb`.


## 1. Load outputs from the main UCI experiment


In [ ]:
# Step 3: The reusable outputs from the correct notebooks are loaded.
# This is done so previous results can be checked before any new model training is considered.

from pathlib import Path
import pandas as pd
import json

BASE_DIR = Path(".")
TABLE_DIR = BASE_DIR / "pipeline_outputs" / "tables"

files_to_check = [
    "feature_group_mapping.csv",
    "exp3_feature_masking_results.csv",
    "exp4_permutation_importance.csv",
    "exp4_shap_importance.csv",
    "exp4_correlation.csv",
    "exp4_mutual_information.csv",
    "exp4_combined_ranking.csv",
    "exp4_progressive_restriction_results.csv",
]

print("Checking reusable files:\n")

for fname in files_to_check:
    path = TABLE_DIR / fname
    
    print("="*80)
    print(fname)
    
    if not path.exists():
        print("File not found")
        continue
    
    df = pd.read_csv(path)
    print(f"✅ Found | shape = {df.shape}")
    print("Columns:", list(df.columns))
    display(df.head())


## 2. Define the Observation group


In [ ]:
# Step 4: Observation variables are extracted from the saved feature mapping.
# This is done to understand what the Observation group actually contains.
# Conor specifically asked for this before any further experiments.

import pandas as pd

mapping = pd.read_csv(
    "pipeline_outputs/tables/feature_group_mapping.csv"
)

obs_vars = mapping[mapping["group"] == "Observation"]

print(f"\nObservation variables: {len(obs_vars)}\n")

display(obs_vars[["col_index", "col_name"]].sort_values("col_index"))


## 3. Variable-level association checks


In [ ]:
# Step 5: Observation variables are ranked by simple association with the target.
# This is done to identify whether one variable dominates the Observation group.
# Conor specifically raised the possibility that only one or two variables may be driving the result.

import pandas as pd
from sklearn.feature_selection import mutual_info_classif

# Load train data used in experiments
X_train = pd.read_csv("pipeline_outputs/tables/X_train_scaled.csv")
y_train = pd.read_csv("pipeline_outputs/tables/y_train.csv").squeeze()

obs_vars = [
    "admission_type_id",
    "discharge_disposition_id",
    "admission_source_id",
    "time_in_hospital",
    "num_lab_procedures",
    "num_procedures",
    "num_medications",
    "number_outpatient",
    "number_emergency",
    "number_inpatient"
]

# Mutual information is calculated for Observation variables only.
mi = mutual_info_classif(
    X_train[obs_vars],
    y_train,
    random_state=42
)

mi_df = pd.DataFrame({
    "Variable": obs_vars,
    "Mutual_Information": mi
}).sort_values(
    "Mutual_Information",
    ascending=False
)

display(mi_df)


In [ ]:
# Step 6: Observation variables are checked individually against the target.
# This is done to identify whether any variable shows suspiciously strong association.
# A leakage pattern would normally appear as one variable dominating all others.

import pandas as pd

X_train = pd.read_csv("pipeline_outputs/tables/X_train_scaled.csv")
y_train = pd.read_csv("pipeline_outputs/tables/y_train.csv").squeeze()

obs_vars = [
    "admission_type_id",
    "discharge_disposition_id",
    "admission_source_id",
    "time_in_hospital",
    "num_lab_procedures",
    "num_procedures",
    "num_medications",
    "number_outpatient",
    "number_emergency",
    "number_inpatient"
]

rows = []

for col in obs_vars:
    
    pearson = X_train[col].corr(y_train)
    
    rows.append({
        "Variable": col,
        "Abs_Pearson": abs(pearson),
        "Pearson": pearson
    })

corr_df = pd.DataFrame(rows)

corr_df = corr_df.sort_values(
    "Abs_Pearson",
    ascending=False
)

display(corr_df)


In [ ]:
# Step 7: Observation variables are ranked by removal impact.
# This is done to determine whether Observation dominance is caused
# by one variable or by the combined contribution of several variables.

import pandas as pd

X_train = pd.read_csv("pipeline_outputs/tables/X_train_scaled.csv")
X_test  = pd.read_csv("pipeline_outputs/tables/X_test_scaled_168.csv")

print("Train shape:", X_train.shape)
print("Test shape :", X_test.shape)

obs_vars = [
    "admission_type_id",
    "discharge_disposition_id",
    "admission_source_id",
    "time_in_hospital",
    "num_lab_procedures",
    "num_procedures",
    "num_medications",
    "number_outpatient",
    "number_emergency",
    "number_inpatient"
]

missing_train = [c for c in obs_vars if c not in X_train.columns]
missing_test  = [c for c in obs_vars if c not in X_test.columns]

print("\nMissing in train:", missing_train)
print("Missing in test :", missing_test)


In [ ]:
# Step 8: Candidate train datasets are inspected.
# This is done to identify the exact train matrix used in the feature-group experiments.

import pandas as pd

candidate_files = [
    "pipeline_outputs/tables/X_train_scaled.csv",
    "pipeline_outputs/tables/X_test_scaled_168.csv",
    "paper_outputs/step4D_X_train.csv",
    "paper_outputs/step4D_X_test.csv",
]

for f in candidate_files:
    
    try:
        df = pd.read_csv(f, nrows=5)
        print("\n" + "="*80)
        print(f)
        print("Shape (5 rows preview):", df.shape)
        print("First 15 columns:")
        print(list(df.columns[:15]))
        
    except Exception as e:
        print(f"\n{f}")
        print("ERROR:", e)


## 4. Single-variable predictive signal


In [ ]:
# Step 9: Each Observation variable is evaluated individually.
# This is done to determine whether one variable can predict readmission on its own.
# If no variable performs strongly alone, Observation importance is likely distributed.

import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

X_train = pd.read_csv("paper_outputs/step4D_X_train.csv")
X_test  = pd.read_csv("paper_outputs/step4D_X_test.csv")

y_train = pd.read_csv("paper_outputs/step4D_y_train.csv").squeeze()
y_test  = pd.read_csv("paper_outputs/step4D_y_test.csv").squeeze()

obs_vars = [
    "admission_type_id",
    "discharge_disposition_id",
    "admission_source_id",
    "time_in_hospital",
    "num_lab_procedures",
    "num_procedures",
    "num_medications",
    "number_outpatient",
    "number_emergency",
    "number_inpatient"
]

results = []

for var in obs_vars:

    model = LogisticRegression(
        max_iter=1000,
        random_state=42
    )

    model.fit(
        X_train[[var]],
        y_train
    )

    probs = model.predict_proba(
        X_test[[var]]
    )[:,1]

    auc = roc_auc_score(
        y_test,
        probs
    )

    results.append({
        "Variable": var,
        "Single_Variable_AUC": round(auc, 4)
    })

results = pd.DataFrame(results)

results = results.sort_values(
    "Single_Variable_AUC",
    ascending=False
)

display(results)


In [ ]:
# Step 10: Each Observation variable is tested as a single predictor across the three main models.
# This is done to check whether any one Observation variable alone can explain the Observation effect.
# A leakage-like variable would show unusually high AUC across models.

import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# Data used in the grouped-feature experiments is loaded.
X_train = pd.read_csv("paper_outputs/step4D_X_train.csv")
X_test  = pd.read_csv("paper_outputs/step4D_X_test.csv")

y_train = pd.read_csv("paper_outputs/step4D_y_train.csv").squeeze()
y_test  = pd.read_csv("paper_outputs/step4D_y_test.csv").squeeze()

obs_vars = [
    "admission_type_id",
    "discharge_disposition_id",
    "admission_source_id",
    "time_in_hospital",
    "num_lab_procedures",
    "num_procedures",
    "num_medications",
    "number_outpatient",
    "number_emergency",
    "number_inpatient"
]

models = {
    "LR": LogisticRegression(
        max_iter=1000,
        random_state=42
    ),
    
    "XGBoost": XGBClassifier(
        n_estimators=100,
        max_depth=3,
        learning_rate=0.05,
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    ),
    
    "LightGBM": LGBMClassifier(
        n_estimators=100,
        max_depth=3,
        learning_rate=0.05,
        random_state=42,
        n_jobs=-1,
        verbose=-1
    )
}

rows = []

for var in obs_vars:
    
    for model_name, model in models.items():
        
        # The model is trained using one Observation variable only.
        # This checks whether that variable alone has suspiciously strong predictive power.
        model.fit(
            X_train[[var]],
            y_train
        )
        
        probs = model.predict_proba(
            X_test[[var]]
        )[:, 1]
        
        auc = roc_auc_score(
            y_test,
            probs
        )
        
        rows.append({
            "Variable": var,
            "Model": model_name,
            "Single_Variable_AUC": round(auc, 4)
        })

single_var_auc = pd.DataFrame(rows)

# Mean AUC across models is added for easier ranking.
summary = (
    single_var_auc
    .groupby("Variable")["Single_Variable_AUC"]
    .agg(["mean", "min", "max"])
    .reset_index()
    .rename(columns={
        "mean": "Mean_AUC",
        "min": "Min_AUC",
        "max": "Max_AUC"
    })
)

summary["Mean_AUC"] = summary["Mean_AUC"].round(4)
summary["Min_AUC"] = summary["Min_AUC"].round(4)
summary["Max_AUC"] = summary["Max_AUC"].round(4)

summary = summary.sort_values(
    "Mean_AUC",
    ascending=False
)

display(single_var_auc)
display(summary)

# Results are saved for the observation-validation notebook/paper table.
single_var_auc.to_csv(
    "observation_single_variable_auc_by_model.csv",
    index=False
)

summary.to_csv(
    "observation_single_variable_auc_summary.csv",
    index=False
)


## 5. Observation-only vs non-Observation models


In [ ]:
# Step 11: Observation-only, Non-Observation-only, and Full models are compared.
# This is done to test whether the Observation group carries the main predictive signal as a group.
# The comparison is repeated across LR, XGBoost, and LightGBM for stronger evidence.

import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, f1_score, average_precision_score

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# The grouped-feature train/test data used in the main experiments is loaded.
X_train = pd.read_csv("paper_outputs/step4D_X_train.csv")
X_test  = pd.read_csv("paper_outputs/step4D_X_test.csv")

y_train = pd.read_csv("paper_outputs/step4D_y_train.csv").squeeze()
y_test  = pd.read_csv("paper_outputs/step4D_y_test.csv").squeeze()

obs_vars = [
    "admission_type_id",
    "discharge_disposition_id",
    "admission_source_id",
    "time_in_hospital",
    "num_lab_procedures",
    "num_procedures",
    "num_medications",
    "number_outpatient",
    "number_emergency",
    "number_inpatient"
]

non_obs_vars = [
    c for c in X_train.columns
    if c not in obs_vars
]

feature_sets = {
    "Observation only": obs_vars,
    "Non-Observation only": non_obs_vars,
    "Full feature set": list(X_train.columns)
}

models = {
    "LR": LogisticRegression(
        max_iter=1000,
        random_state=42
    ),
    
    "XGBoost": XGBClassifier(
        n_estimators=100,
        max_depth=3,
        learning_rate=0.05,
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    ),
    
    "LightGBM": LGBMClassifier(
        n_estimators=100,
        max_depth=3,
        learning_rate=0.05,
        random_state=42,
        n_jobs=-1,
        verbose=-1
    )
}

rows = []

for feature_set_name, cols in feature_sets.items():
    
    for model_name, model in models.items():
        
        # The model is trained using one feature setting at a time.
        # This isolates the predictive contribution of Observation versus all other groups.
        model.fit(
            X_train[cols],
            y_train
        )
        
        probs = model.predict_proba(
            X_test[cols]
        )[:, 1]
        
        preds = (probs >= 0.5).astype(int)
        
        rows.append({
            "Feature_Set": feature_set_name,
            "Model": model_name,
            "N_Features": len(cols),
            "AUC": round(roc_auc_score(y_test, probs), 4),
            "Average_Precision": round(average_precision_score(y_test, probs), 4),
            "F1_at_0.5": round(f1_score(y_test, preds), 4)
        })

obs_vs_nonobs_results = pd.DataFrame(rows)

display(obs_vs_nonobs_results)

# A pivot table is created for easier comparison in the paper.
auc_table = obs_vs_nonobs_results.pivot(
    index="Model",
    columns="Feature_Set",
    values="AUC"
).reset_index()

display(auc_table)

# Results are saved for later use in the paper.
obs_vs_nonobs_results.to_csv(
    "observation_vs_nonobservation_model_results.csv",
    index=False
)

auc_table.to_csv(
    "observation_vs_nonobservation_auc_table.csv",
    index=False
)


## 6. Leave-one-variable-out ablation


In [ ]:
# Step 12: Observation-variable ablation analysis is performed.
# One Observation variable is removed at a time while all other features are retained.
# This is done to identify whether Observation dominance is driven by one variable
# or emerges from the combined contribution of multiple healthcare utilisation indicators.

import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score

# Data used in the main grouped-feature experiments is loaded.
X_train = pd.read_csv("paper_outputs/step4D_X_train.csv")
X_test  = pd.read_csv("paper_outputs/step4D_X_test.csv")

y_train = pd.read_csv("paper_outputs/step4D_y_train.csv").squeeze()
y_test  = pd.read_csv("paper_outputs/step4D_y_test.csv").squeeze()

obs_vars = [
    "admission_type_id",
    "discharge_disposition_id",
    "admission_source_id",
    "time_in_hospital",
    "num_lab_procedures",
    "num_procedures",
    "num_medications",
    "number_outpatient",
    "number_emergency",
    "number_inpatient"
]

# Baseline model is trained using all features.
baseline_model = LGBMClassifier(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.05,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

baseline_model.fit(X_train, y_train)

baseline_probs = baseline_model.predict_proba(X_test)[:,1]

baseline_auc = roc_auc_score(
    y_test,
    baseline_probs
)

print(f"Baseline AUC = {baseline_auc:.4f}")

rows = []

for var in obs_vars:

    remaining_cols = [
        c for c in X_train.columns
        if c != var
    ]

    model = LGBMClassifier(
        n_estimators=100,
        max_depth=3,
        learning_rate=0.05,
        random_state=42,
        n_jobs=-1,
        verbose=-1
    )

    model.fit(
        X_train[remaining_cols],
        y_train
    )

    probs = model.predict_proba(
        X_test[remaining_cols]
    )[:,1]

    auc = roc_auc_score(
        y_test,
        probs
    )

    rows.append({
        "Removed_Variable": var,
        "AUC": round(auc,4),
        "AUC_Drop": round(
            baseline_auc - auc,
            4
        )
    })

ablation_results = pd.DataFrame(rows)

ablation_results = ablation_results.sort_values(
    "AUC_Drop",
    ascending=False
)

display(ablation_results)

ablation_results.to_csv(
    "observation_variable_ablation_lgbm.csv",
    index=False
)


## 7. Top-2 vs remaining Observation variables


In [ ]:
# Step 13: Observation-group decomposition is repeated across the three main models.
# This is done to check whether the Top-2 Observation explanation is consistent across model families.

import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

X_train = pd.read_csv("paper_outputs/step4D_X_train.csv")
X_test  = pd.read_csv("paper_outputs/step4D_X_test.csv")

y_train = pd.read_csv("paper_outputs/step4D_y_train.csv").squeeze()
y_test  = pd.read_csv("paper_outputs/step4D_y_test.csv").squeeze()

all_obs = [
    "admission_type_id",
    "discharge_disposition_id",
    "admission_source_id",
    "time_in_hospital",
    "num_lab_procedures",
    "num_procedures",
    "num_medications",
    "number_outpatient",
    "number_emergency",
    "number_inpatient"
]

top2_obs = [
    "number_inpatient",
    "discharge_disposition_id"
]

remaining_obs = [
    c for c in all_obs
    if c not in top2_obs
]

feature_sets = {
    "Full model": list(X_train.columns),
    "All Observation": all_obs,
    "Top-2 Observation": top2_obs,
    "Remaining 8 Observation": remaining_obs
}

models = {
    "LR": LogisticRegression(
        max_iter=5000,
        solver="lbfgs",
        random_state=42
    ),
    
    "XGBoost": XGBClassifier(
        n_estimators=100,
        max_depth=3,
        learning_rate=0.05,
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    ),
    
    "LightGBM": LGBMClassifier(
        n_estimators=100,
        max_depth=3,
        learning_rate=0.05,
        random_state=42,
        n_jobs=-1,
        verbose=-1
    )
}

rows = []

for feature_set_name, cols in feature_sets.items():
    
    for model_name, model in models.items():
        
        # Each model is trained using one feature set at a time.
        # This isolates whether performance comes from all Observation variables,
        # only the Top-2 variables, or the remaining Observation variables.
        model.fit(
            X_train[cols],
            y_train
        )
        
        probs = model.predict_proba(
            X_test[cols]
        )[:, 1]
        
        rows.append({
            "Feature_Set": feature_set_name,
            "Model": model_name,
            "N_Features": len(cols),
            "AUC": round(roc_auc_score(y_test, probs), 4),
            "Average_Precision": round(average_precision_score(y_test, probs), 4)
        })

decomposition_results = pd.DataFrame(rows)

display(decomposition_results)

# AUC is reshaped into a paper-friendly table.
decomposition_auc_table = decomposition_results.pivot(
    index="Feature_Set",
    columns="Model",
    values="AUC"
).reset_index()

# Feature-set order is fixed for easier reading.
feature_order = [
    "Full model",
    "All Observation",
    "Top-2 Observation",
    "Remaining 8 Observation"
]

decomposition_auc_table["Feature_Set"] = pd.Categorical(
    decomposition_auc_table["Feature_Set"],
    categories=feature_order,
    ordered=True
)

decomposition_auc_table = decomposition_auc_table.sort_values("Feature_Set")

display(decomposition_auc_table)

decomposition_results.to_csv(
    "observation_group_decomposition_all_models.csv",
    index=False
)

decomposition_auc_table.to_csv(
    "observation_group_decomposition_auc_table.csv",
    index=False
)
